# IntentClassifier Evaluation Runner

Interactive notebook for running and analyzing IntentClassifier evaluations using the evaluation framework.

The IntentClassifier classifies user messages into three dimensions:
- **Task**: consult (discovery) or reflect (analysis)
- **Mode**: reporter / interpreter / explorer (response depth)
- **Job**: workflow type (query, visualize, discover, etc.)

## Framework Features
- **Provenance tracking**: Git commit, prompt hash, dataset hash
- **Results storage**: SQLite index + Parquet details
- **CLI integration**: `just eval intent_classifier`

In [1]:
import matplotlib.pyplot as plt
import nest_asyncio
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from agents import enable_cache

# Evaluation framework
from agents.evals import (
    Experiment,
    ExperimentSet,
    classification_metrics_from_report,
    metrics,
)
from agents.evals.datasets.intent_classifier import (
    ALL_CASES,
    ALL_RAW_CASES,
    CONSULT_CASES,
    EDGE_CASES,
    REFLECT_CASES,
    SHOW_REVENUE,
    VAGUE_QUERY,
    WHY_CHURN_RISE,
)
from agents.intent_classifier import IntentClassifier
from agents.intent_classifier.schemas import IntentClassifierInput

# Handle Jupyter async context
nest_asyncio.apply()

# Enable LLM caching for faster reruns
enable_cache()

In [2]:
# Default configuration
DEFAULT_MODEL = "google:gemini-3-flash-preview"
PROMPT_REF = "py://agents.intent_classifier.agent:DEFAULT_PROMPT"

# Create agent for single-case testing (uses default prompt)
agent = IntentClassifier(model=DEFAULT_MODEL)

# Or experiment with a custom prompt:
# agent = IntentClassifier(prompt_override="Your custom prompt here...")

## Single Case Testing

Test individual cases for debugging and prompt iteration.

In [3]:
# Run Single Case (debugging)
result = await agent.arun(SHOW_REVENUE.inputs)

print(f"Input: {SHOW_REVENUE.inputs.message_text}\n")
print(result)
print(
    f"\nExpected: task={SHOW_REVENUE.expected_output.task}, mode={SHOW_REVENUE.expected_output.mode}, job={SHOW_REVENUE.expected_output.job}"
)

Input: Show Q4 revenue by region

  Task:       consult
  Mode:       reporter
  Job:        query
  Confidence: 100%
  Rationale:  Direct request for revenue data filtered by a specific time period and grouped by region.
  Metrics:    revenue
  Dimensions: region
  Filters:    {'quarter': 'Q4'}
  Time range: Q4

Expected: task=consult, mode=reporter, job=query


In [10]:
# Test a reflect/interpreter case
result = await agent.arun(WHY_CHURN_RISE.inputs)

print(f"Input: {WHY_CHURN_RISE.inputs.message_text}\n")
print(result)
print(
    f"\nExpected: task={WHY_CHURN_RISE.expected_output.task}, mode={WHY_CHURN_RISE.expected_output.mode}, job={WHY_CHURN_RISE.expected_output.job}"
)

Input: Why did churn rise despite stable pricing?

  Task:       reflect
  Mode:       interpreter
  Job:        query
  Confidence: 92%
  Rationale:  The user is asking for an explanation (why) regarding a specific metric change (churn rise) in the context of another factor (stable pricing), which requires analysis and interpretation of existing data patterns.
  Metrics:    churn, pricing

Expected: task=reflect, mode=interpreter, job=query


In [7]:
# Test a clarification case
result = await agent.arun(VAGUE_QUERY.inputs)

print(f"Input: {VAGUE_QUERY.inputs.message_text}\n")
print(result)
print(f"\nExpected: job={VAGUE_QUERY.expected_output.job}")

Input: show me the data

  Task:       consult
  Mode:       reporter
  Job:        clarify
  Confidence: 40%
  Rationale:  The user is asking for data but has not specified a metric, domain, or time range, making the request too vague to execute.
  Clarification: I would be happy to show you some data. Could you please specify what you are interested in? For example, are you looking for Economic indicators (like GDP), Corporate financial statements, or Agricultural production data?

Expected: job=clarify


## Ad-hoc Testing

Test arbitrary queries interactively.

In [8]:
# Test your own query
test_input = IntentClassifierInput(
    message_text="What's driving the margin improvement this quarter?"
)

result = await agent.arun(test_input)
print(result)

  Task:       reflect
  Mode:       interpreter
  Job:        query
  Confidence: 90%
  Rationale:  The user is asking for the causes behind a change in a metric (margin), which constitutes diagnostic analysis.
  Metrics:    margin
  Time range: this quarter


## Full Evaluation with Framework

Run all test cases using the evaluation framework. This provides:
- Per-field match metrics (task, mode, job)
- Provenance tracking (git commit, prompt hash)
- Optional result storage for comparison

In [9]:
# Run full evaluation with framework
exp = Experiment(
    agent=IntentClassifier,
    prompt=PROMPT_REF,
    model=DEFAULT_MODEL,
    dataset=ALL_CASES,
    dataset_name="ALL_CASES",
    metrics=[metrics.classification(fields=["task", "mode", "job"])],
)

# Run without saving (set save=True to persist results)
result = await exp.run(save=False)

# Print summary
result.print_summary()

Output()

Experiment: IntentClassifier_gemini-3-flash-preview
Model: google:gemini-3-flash-preview
Git: 89822d5c[dirty]
Cases: 14 (0 failed)
Duration: 15.3s

Success rate: 100.0%


In [ ]:
# Detailed sklearn classification metrics (requires raw pydantic-evals report)
# For deeper analysis, run a raw evaluation to get the full report
from pydantic_evals import Dataset

dataset = Dataset(cases=ALL_CASES)  # ALL_CASES already contains Case objects
report = await dataset.evaluate(agent.arun, max_concurrency=5)

# Classification metrics per dimension
for dimension in ["task", "mode", "job"]:
    dim_metrics = classification_metrics_from_report(ALL_RAW_CASES, report, dimension)
    dim_metrics.print_report()
    print(f"Macro F1: {dim_metrics.macro_f1():.3f}\n")

## Multi-Model Comparison

Compare IntentClassifier performance across different LLM providers and models using ExperimentSet.

In [4]:
# Models to compare
MODELS_TO_COMPARE = [
    "google:gemini-2.5-flash-lite",
    "google:gemini-3-flash-preview",
    # "anthropic:claude-sonnet-4-20250514",
    # "openai:gpt-4o-mini",
]

# Create experiment set for comparison
exp_set = ExperimentSet(
    name="IntentClassifier Model Comparison",
    experiments=[
        Experiment(
            agent=IntentClassifier,
            prompt=PROMPT_REF,
            model=model,
            dataset=ALL_CASES,
            dataset_name="ALL_CASES",
            metrics=[metrics.classification(fields=["task", "mode", "job"])],
        )
        for model in MODELS_TO_COMPARE
    ],
)

print(exp_set)

IntentClassifier Model Comparison (2 experiments):
  - gemini-2.5-flash-lite
  - gemini-3-flash-preview


In [5]:
# Run comparison
comparison = await exp_set.run(save=False)
comparison.print_comparison()

# Store results for visualization
model_results = comparison.results

Evaluating 14 cases with gemini-2.5-flash-lite... done (2.6s)
Evaluating 14 cases with gemini-3-flash-preview... done (22.6s)

Comparison: IntentClassifier Model Comparison

gemini-2.5-flash-lite (14 cases, 2.6s)
  Field    | Accuracy |       F1 | Precision |   Recall
  -------- | -------- | -------- | --------- | --------
  job      |    92.9% |    92.0% |     94.0% |    92.9%
  mode     |    85.7% |    85.4% |     88.9% |    85.7%
  task     |    85.7% |    85.7% |     85.7% |    85.7%

gemini-3-flash-preview (14 cases, 22.6s)
  Field    | Accuracy |       F1 | Precision |   Recall
  -------- | -------- | -------- | --------- | --------
  job      |    92.9% |    93.2% |     95.2% |    92.9%
  mode     |    85.7% |    85.4% |     88.9% |    85.7%
  task     |    85.7% |    85.7% |     85.7% |    85.7%


In [6]:
# DataFrame view of comparison results
display(comparison.to_dataframe())

,name,model,prompt_hash,success_rate,task_accuracy,mode_accuracy,job_accuracy,field_accuracy,task_f1,task_precision,task_recall,mode_f1,mode_precision,mode_recall,job_f1,job_precision,job_recall,duration_s
0,IntentClassifier_gemini-2.5-flash-lite,google:gemini-2.5-flash-lite,b1e9005b736b,1.0,0.857143,0.857143,0.928571,0.880952,0.857143,0.857143,0.857143,0.853827,0.888889,0.857143,0.919913,0.940476,0.928571,2.782387
1,IntentClassifier_gemini-3-flash-preview,google:gemini-3-flash-preview,b1e9005b736b,1.0,0.857143,0.785714,0.928571,0.857143,0.857143,0.857143,0.857143,0.785714,0.803571,0.785714,0.931746,0.952381,0.928571,17.402308


In [ ]:
# Visualize per-field match rates across models
comparison.plot_metrics(fields=["task_match", "mode_match", "job_match"])
plt.show()

In [ ]:
# Speed vs accuracy tradeoff
comparison.plot_speed_accuracy()
plt.show()

## Subset Evaluation

Evaluate specific categories of test cases to identify weaknesses.

In [ ]:
# Evaluate only Consult cases
consult_exp = Experiment(
    agent=IntentClassifier,
    prompt=PROMPT_REF,
    model=DEFAULT_MODEL,
    dataset=CONSULT_CASES,
    dataset_name="CONSULT_CASES",
    metrics=[metrics.classification(fields=["task", "mode", "job"])],
)
consult_result = await consult_exp.run(save=False)

print(f"Consult Cases Success Rate: {consult_result.success_rate:.1%}")
print(f"Total Consult Cases: {consult_result.total_cases}")

In [ ]:
# Evaluate only Reflect cases
reflect_exp = Experiment(
    agent=IntentClassifier,
    prompt=PROMPT_REF,
    model=DEFAULT_MODEL,
    dataset=REFLECT_CASES,
    dataset_name="REFLECT_CASES",
    metrics=[metrics.classification(fields=["task", "mode", "job"])],
)
reflect_result = await reflect_exp.run(save=False)

print(f"Reflect Cases Success Rate: {reflect_result.success_rate:.1%}")
print(f"Total Reflect Cases: {reflect_result.total_cases}")

In [ ]:
# Evaluate edge cases (clarification, out_of_scope)
edge_exp = Experiment(
    agent=IntentClassifier,
    prompt=PROMPT_REF,
    model=DEFAULT_MODEL,
    dataset=EDGE_CASES,
    dataset_name="EDGE_CASES",
    metrics=[metrics.classification(fields=["task", "mode", "job"])],
)
edge_result = await edge_exp.run(save=False)

print(f"Edge Cases Success Rate: {edge_result.success_rate:.1%}")
print(f"Total Edge Cases: {edge_result.total_cases}")
print("\nPer-field metrics:")
for key, value in edge_result.summary.items():
    if key.endswith("_accuracy") or key.endswith("_f1"):
        print(f"  {key}: {value:.1%}")

## Confusion Matrix Visualization

Visualize misclassifications per dimension using the sklearn report from the full evaluation above.

In [ ]:
# Task confusion matrix (uses report from cell above)
task_metrics = classification_metrics_from_report(ALL_RAW_CASES, report, "task")
cm = confusion_matrix(
    task_metrics.y_true, task_metrics.y_pred, labels=["consult", "reflect"]
)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["consult", "reflect"]
)
disp.plot()
plt.title("Task Confusion Matrix")
plt.show()

In [ ]:
# Mode confusion matrix
mode_metrics = classification_metrics_from_report(ALL_RAW_CASES, report, "mode")
cm = confusion_matrix(
    mode_metrics.y_true,
    mode_metrics.y_pred,
    labels=["reporter", "interpreter", "explorer"],
)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["reporter", "interpreter", "explorer"]
)
disp.plot()
plt.title("Mode Confusion Matrix")
plt.show()

In [ ]:
# Job confusion matrix (may be sparse with few test cases per job)
job_metrics = classification_metrics_from_report(ALL_RAW_CASES, report, "job")
unique_jobs = sorted(set(job_metrics.y_true + job_metrics.y_pred))
cm = confusion_matrix(job_metrics.y_true, job_metrics.y_pred, labels=unique_jobs)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=unique_jobs)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Job Confusion Matrix")
plt.tight_layout()
plt.show()